# Build RGB and IRG VRTs from a Web Page of COG Links

This notebook takes the URL of a web page that lists Cloud Optimized GeoTIFF files, extracts the COG URLs from that page, and creates two VRT files:

- `RGB`: natural color, using bands `1, 2, 3`
- `IRG`: color infrared style, using bands `4, 1, 2`

A VRT is a small XML file that points to imagery without copying the imagery itself. In this workflow, the VRT points to remote COG files through GDAL's `/vsicurl/` virtual file system, which lets GDAL stream only the pieces of each image it needs over HTTPS.

Dataset used in the default example: NAIP-derived COG imagery served from a Stanford-hosted web directory for the Creek Fire 2020 postfire area. NAIP imagery is aerial imagery collected by the USDA National Agriculture Imagery Program and commonly includes red, green, blue, and near-infrared bands.

## 1. Install and Import Libraries

This notebook uses:

- `requests` to download the HTML page that contains links to COG files.
- `beautifulsoup4` to parse that HTML and find the linked TIFF files.
- `GDAL` through `osgeo.gdal` to create VRT files from the remote COG sources.

GDAL is usually best installed with conda or mamba because it includes compiled geospatial libraries. The installation cell below installs only the lightweight HTML-parsing libraries. If `from osgeo import gdal` fails, install GDAL in your environment before continuing.

In [1]:
# Install the lightweight libraries used to request and parse the web page.
# GDAL is intentionally not installed with pip here because GDAL Python bindings
# must match the compiled GDAL library available in the active environment.
%pip install requests beautifulsoup4

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [beautifulsoup4]m [certifi]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote

import requests
from bs4 import BeautifulSoup
from osgeo import gdal

# Ask GDAL to raise Python exceptions when something goes wrong. This makes
# errors easier for students to see and debug than silent failures.
gdal.UseExceptions()


## 2. Configure the Page URL and Output Filenames

Edit `PAGE_URL` for a different web directory or index page. The default filenames are inferred from this example URL:

`https://web.stanford.edu/~maples/mortalitree/creek_2020/cog/postfire/`

For that page, the default outputs are:

- `creek_2020_postfire_rgb.vrt`
- `creek_2020_postfire_irg.vrt`

To choose your own names, replace `None` with a filename string such as `"my_rgb.vrt"`.

In [53]:
# ----------------------------- User Configuration -----------------------------
# Web page that contains links to COG files. The page can be a simple directory
# listing or any HTML page with links to .tif or .tiff files.
PAGE_URL = "https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/"

# Output folder for the VRT files. Path('.') means the notebook's current
# working directory, which is usually the folder where the notebook is running.
OUTPUT_DIR = Path(".")

# Optional custom output names. Leave as None to infer names from PAGE_URL.
RGB_VRT_FILENAME = "castle_2020_prefire_rgb.vrt"  # Example: "castle_2020_prefire_rgb.vrt"
IRG_VRT_FILENAME = "castle_2020_prefire_irg.vrt"  # Example: "castle_2020_prefire_irg.vrt"

# Band order for each visualization product.
# NAIP-style 4-band imagery is commonly ordered as Red, Green, Blue, NIR.
RGB_BANDS = [1, 2, 3]
IRG_BANDS = [4, 1, 2]

# If True, replace existing VRT files with the same names.
OVERWRITE_OUTPUT = True

# File extensions treated as COG/TIFF sources on the page.
COG_EXTENSIONS = (".tif", ".tiff")


## 3. Define Helper Functions

The helpers below do four jobs:

- infer useful output names from the page URL
- extract COG links from the HTML page
- convert normal HTTPS URLs into GDAL `/vsicurl/` paths
- build a VRT with a specific band order

![Placeholder: Screenshot of a simple web directory listing with several linked .tif files. The image should show that the notebook reads the links from the page rather than requiring students to type each file URL by hand.](placeholder-web-directory-listing-with-cog-links.png)

In [54]:
def infer_output_stem(page_url: str) -> str:
    """Infer a readable output stem such as creek_2020_postfire from a URL."""
    parsed = urlparse(page_url)

    # Split the URL path into meaningful pieces and remove empty pieces caused
    # by leading or trailing slashes.
    path_parts = [unquote(part) for part in parsed.path.split("/") if part]

    # For paths like /mortalitree/creek_2020/cog/postfire/, use the folder
    # before "cog" as the project name and the folder after "cog" as the phase.
    if "cog" in path_parts:
        cog_index = path_parts.index("cog")
        project = path_parts[cog_index - 1] if cog_index > 0 else "cog_sources"
        phase = path_parts[cog_index + 1] if cog_index + 1 < len(path_parts) else "cog"
        return f"{project}_{phase}"

    # Fallback: use the final folder or filename stem from the URL.
    if path_parts:
        return Path(path_parts[-1]).stem or "cog_sources"

    return "cog_sources"


def default_vrt_paths(page_url: str, output_dir: Path) -> tuple[Path, Path]:
    """Create default RGB and IRG VRT paths from the page URL."""
    stem = infer_output_stem(page_url)
    return output_dir / f"{stem}_rgb.vrt", output_dir / f"{stem}_irg.vrt"


def strip_url_query_and_fragment(url: str) -> str:
    """Return a URL without query parameters or fragment text for extension checks."""
    parsed = urlparse(url)
    return parsed._replace(query="", fragment="").geturl()


def find_cog_urls(page_url: str) -> list[str]:
    """Download an HTML page and return absolute URLs for linked TIFF/COG files."""
    response = requests.get(page_url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    cog_urls = []

    for link in soup.find_all("a", href=True):
        # urljoin converts relative links such as image.tif into full URLs
        # using the page URL as the base.
        absolute_url = urljoin(page_url, link["href"])
        clean_url = strip_url_query_and_fragment(absolute_url)

        if clean_url.lower().endswith(COG_EXTENSIONS):
            cog_urls.append(clean_url)

    # Remove duplicates while preserving the original page order. Page order is
    # useful because it often reflects the server's directory listing order.
    return list(dict.fromkeys(cog_urls))


def to_vsicurl(url: str) -> str:
    """Convert an HTTPS URL into the /vsicurl/ path syntax GDAL uses."""
    if url.startswith("/vsicurl/"):
        return url
    return f"/vsicurl/{url}"


def build_band_vrt(source_urls: list[str], output_vrt: Path, bands: list[int]) -> Path:
    """Build one VRT from many remote COG URLs using the requested band order."""
    if not source_urls:
        raise ValueError("No source COG URLs were provided.")

    output_vrt.parent.mkdir(parents=True, exist_ok=True)

    if output_vrt.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"{output_vrt} already exists. Set OVERWRITE_OUTPUT = True or choose a new filename."
        )

    vsicurl_sources = [to_vsicurl(url) for url in source_urls]

    # gdal.BuildVRT creates a virtual mosaic. The bandList argument tells GDAL
    # which source bands should appear in the output VRT, and in what order.
    dataset = gdal.BuildVRT(
        str(output_vrt),
        vsicurl_sources,
        options=gdal.BuildVRTOptions(
            bandList=bands,
            resolution="highest",
            allowProjectionDifference=True,
        ),
    )

    if dataset is None:
        raise RuntimeError(f"GDAL could not build VRT: {output_vrt}")

    # FlushCache forces GDAL to write the VRT XML to disk before we continue.
    dataset.FlushCache()
    dataset = None
    return output_vrt


## 4. Discover the COG URLs on the Page

This cell reads the page, finds linked TIFF files, and prints a short inventory. No imagery is downloaded here. Only the HTML page is downloaded.

In [55]:
cog_urls = find_cog_urls(PAGE_URL)

default_rgb_vrt, default_irg_vrt = default_vrt_paths(PAGE_URL, OUTPUT_DIR)
rgb_vrt = OUTPUT_DIR / RGB_VRT_FILENAME if RGB_VRT_FILENAME else default_rgb_vrt
irg_vrt = OUTPUT_DIR / IRG_VRT_FILENAME if IRG_VRT_FILENAME else default_irg_vrt

print(f"Page URL: {PAGE_URL}")
print(f"COG links found: {len(cog_urls)}")
print(f"RGB VRT output: {rgb_vrt}")
print(f"IRG VRT output: {irg_vrt}")

if not cog_urls:
    raise ValueError("No .tif or .tiff links were found on the page.")

print("\nFirst few COG URLs:")
for url in cog_urls[:5]:
    print(f"- {url}")

if len(cog_urls) > 5:
    print(f"... {len(cog_urls) - 5} more")


Page URL: https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/
COG links found: 43
RGB VRT output: castle_2020_prefire_rgb.vrt
IRG VRT output: castle_2020_prefire_irg.vrt

First few COG URLs:
- https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/m_3611835_se_11_060_20200802.tif
- https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/m_3611835_sw_11_060_20200802.tif
- https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/m_3611837_se_11_060_20200802.tif
- https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/m_3611837_sw_11_060_20200802.tif
- https://web.stanford.edu/~evrobert/mortalitree/castle_2020/naip/cog/prefire/m_3611842_ne_11_060_20200802.tif
... 38 more


## 5. Build the RGB and IRG VRT Files

This cell writes two small VRT files. The VRTs point back to the remote COG URLs, so the COG files themselves are not downloaded.

Conceptually:

- The RGB VRT maps source bands `1, 2, 3` into display bands `red, green, blue`.
- The IRG VRT maps source bands `4, 1, 2` into display bands `near infrared, red, green`.
- Healthy vegetation often appears bright in the near-infrared band, so the IRG view can make vegetation patterns easier to see than natural color alone.

![Placeholder: Side-by-side example showing the same NAIP scene as natural color RGB and false-color IRG. The IRG side should emphasize vegetation response with bright red or pink tones.](placeholder-rgb-vs-irg-naip-comparison.png)

In [56]:
rgb_output = build_band_vrt(cog_urls, rgb_vrt, RGB_BANDS)
irg_output = build_band_vrt(cog_urls, irg_vrt, IRG_BANDS)

print("VRT files created:")
print(f"- RGB: {rgb_output.resolve()}")
print(f"- IRG: {irg_output.resolve()}")


VRT files created:
- RGB: /Users/maples/GitHub/EarthExplorer_API/notebooks/castle_2020_prefire_rgb.vrt
- IRG: /Users/maples/GitHub/EarthExplorer_API/notebooks/castle_2020_prefire_irg.vrt


## 6. Optional: Inspect the VRTs with GDAL

This quick check opens each VRT and reports its size and band count. If the VRT opens successfully, GDAL can read the VRT structure and reach the remote COG sources.

In [57]:
def summarize_vrt(vrt_path: Path) -> None:
    """Open a VRT and print basic raster metadata."""
    dataset = gdal.Open(str(vrt_path))
    if dataset is None:
        raise RuntimeError(f"Could not open VRT: {vrt_path}")

    print(f"{vrt_path.name}")
    print(f"  size: {dataset.RasterXSize} columns x {dataset.RasterYSize} rows")
    print(f"  bands: {dataset.RasterCount}")
    print(f"  projection available: {bool(dataset.GetProjection())}")
    dataset = None


summarize_vrt(rgb_vrt)
summarize_vrt(irg_vrt)


castle_2020_prefire_rgb.vrt
  size: 94410 columns x 71040 rows
  bands: 3
  projection available: True
castle_2020_prefire_irg.vrt
  size: 94410 columns x 71040 rows
  bands: 3
  projection available: True
